# Oasis Infobyte – Data Cleaning Task

Clean the dirty Retail Store Sales dataset, recover logically inferable values, validate transaction totals, and export a clean dataset.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

df = pd.read_csv('../data/retail_store_sales.csv')
print('Shape:', df.shape)
display(df.head())

## 1. Data quality inspection

In [ ]:
display(df.info())
display(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())

## 2. Standardize values and data types

In [ ]:
null_markers = {'', 'None', 'none', 'NULL', 'null', 'N/A', 'n/a', 'UNKNOWN', 'unknown', 'ERROR', 'error'}
for c in df.columns:
    if df[c].dtype == 'object':
        df[c] = df[c].astype('string').str.strip().mask(df[c].astype('string').str.strip().isin(null_markers))
for c in ['Price Per Unit','Quantity','Total Spent']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')

## 3. Remove exact duplicates

In [ ]:
before = len(df)
df = df.drop_duplicates().copy()
print('Duplicates removed:', before-len(df))

## 4. Recover missing transaction values

In [ ]:
m = df['Quantity'].isna() & df['Price Per Unit'].notna() & df['Total Spent'].notna() & df['Price Per Unit'].ne(0)
df.loc[m,'Quantity'] = df.loc[m,'Total Spent'] / df.loc[m,'Price Per Unit']

m = df['Price Per Unit'].isna() & df['Quantity'].notna() & df['Total Spent'].notna() & df['Quantity'].ne(0)
df.loc[m,'Price Per Unit'] = df.loc[m,'Total Spent'] / df.loc[m,'Quantity']

expected = df['Quantity'] * df['Price Per Unit']
m = df['Total Spent'].isna() & expected.notna()
df.loc[m,'Total Spent'] = expected[m]

# Correct inconsistent totals
expected = df['Quantity'] * df['Price Per Unit']
bad = df['Total Spent'].notna() & expected.notna() & ~np.isclose(df['Total Spent'], expected, atol=0.01, rtol=0)
df.loc[bad,'Total Spent'] = expected[bad]
print('Remaining missing numeric values:', df[['Quantity','Price Per Unit','Total Spent']].isna().sum().sum())

## 5. Infer missing Item values

In [ ]:
lookup = (df.dropna(subset=['Item','Category','Price Per Unit'])
            .drop_duplicates(['Category','Price Per Unit'])
            [['Category','Price Per Unit','Item']])
price_to_item = {(r['Category'],r['Price Per Unit']):r['Item'] for _,r in lookup.iterrows()}
mask = df['Item'].isna() & df['Category'].notna() & df['Price Per Unit'].notna()
inferred = df.loc[mask].apply(lambda r: price_to_item.get((r['Category'],r['Price Per Unit'])), axis=1)
valid = inferred.notna()
df.loc[inferred.index[valid],'Item'] = inferred[valid]
print('Items inferred:', valid.sum())

## 6. Handle remaining missing values and validate

In [ ]:
for c in ['Quantity','Price Per Unit','Total Spent']:
    if df[c].isna().any(): df[c] = df[c].fillna(df[c].median())
for c in ['Item','Payment Method','Location']:
    if df[c].isna().any(): df[c] = df[c].fillna(df[c].mode(dropna=True).iloc[0])
if 'Discount Applied' in df.columns:
    df['Discount Applied'] = df['Discount Applied'].astype('boolean').fillna(False)

df['Price Per Unit'] = df['Price Per Unit'].round(2)
df['Total Spent'] = df['Total Spent'].round(2)
print('Missing cells:', df.isna().sum().sum())
print('Duplicates:', df.duplicated().sum())
print('Formula mismatches:', (~np.isclose(df['Total Spent'],df['Quantity']*df['Price Per Unit'],atol=0.01,rtol=0)).sum())

In [ ]:
Path('../output').mkdir(exist_ok=True)
df.to_csv('../output/retail_store_sales_cleaned.csv', index=False)
print('Saved cleaned dataset.')